In [ ]:
import json
import core.similaritysearch as simsearch

import os

import datetime
from numpyencoder import NumpyEncoder
from pathlib import Path

# root directory path
ROOT = Path(os.getcwd()).resolve().parents[0]

In [2]:
date = datetime.datetime.now().strftime("%y-%m-%d_%H:%M")
date

'25-09-26_15:50'

In [3]:
relevant_page_texts = {}
relevant_page_texts["lieder"] = []
relevant_page_texts["bibel"] = []

for n in range(41, 1291):
    with open(ROOT / f"source_texts/praxis_pietatis_verses/{n}.json") as f:
        page = json.load(f)
    page_info = {}
    page_info[n] = page
    relevant_page_texts["lieder"].append(page_info)

with open(ROOT / "source_texts/bible/old_testament_chunked.json") as f:
    old_testament = json.load(f)
for book_name, chapters in old_testament.items():
    for chapter, verses in chapters.items():
        for verse_nr, verse in verses.items():
            verse_info = {}
            verse_id = f"{book_name}_{chapter}_{verse_nr}"
            verse_info[verse_id] = verse
            relevant_page_texts["bibel"].append(verse_info)
            
with open(ROOT / f"source_texts/bible/new_testament_chunked.json") as f:
    new_testament = json.load(f)
for book_name, chapters in new_testament.items():
    for chapter, verses in chapters.items():
        for verse_nr, verse in verses.items():
            verse_info = {}
            verse_id = f"{book_name}_{chapter}_{verse_nr}"
            verse_info[verse_id] = verse
            relevant_page_texts["bibel"].append(verse_info)

In [4]:
with open(ROOT / "predigten_übersicht.json") as f:
    predigten = json.load(f)

In [5]:
sermons = list(predigten.keys())

In [6]:
sermons = ["E000036"]
#sermons = sermons[:14]
#sermons = sermons[30:]

In [11]:
fuzziness = 50

In [8]:
similarity_table = {}
similarity_table['date'] = date
similarity_table['corpus'] = "all_sermons"
similarity_table['method'] = 'similarity_search'
similarity_table['fuzziness'] = fuzziness


In [12]:
for sermon in sermons:
    for x in ["bibel", "lieder"]:
        hits = simsearch.find_similarities(x, sermon, relevant_page_texts[x], fuzziness)
        print("Starting with Corrections")
        hits = simsearch.correct_inbetween_matches(hits)
        print("Starting with inferrences")
        hits = simsearch.add_inferred_matches(hits, sermon)
        
        filepath = f"predictions/{sermon}_{x}_{similarity_table['method']}_{similarity_table['fuzziness']}_{similarity_table['date']}.csv"

        hits.to_csv(ROOT / filepath, index=False)

        info = {}
        info["date"] = similarity_table['date']
        info["task"] = x
        info["method"] = similarity_table['method']
        info["fuzziness"] = similarity_table['fuzziness']
        info['file'] = str(filepath)

        # append metadata if file already exists,
        # otherwise create new file
        my_file = Path(ROOT / f"predictions/{sermon}_predictions.json")
        if my_file.is_file():
            with open(my_file, "r", encoding="utf-8") as f:
                predictions = json.load(f)
            predictions.append(info)
            with open(my_file, "w", encoding="utf-8") as f:
                json.dump(predictions, f, ensure_ascii=False)
        else:
            with open(ROOT / f"predictions/{sermon}_predictions.json", "x", encoding="utf-8") as f:
                json.dump([info], f, ensure_ascii=False)




Starting with E000036 (bibel), fuzziness: 50


KeyboardInterrupt: 